In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/phhasian0710/seed-iv/Channel Order.xlsx
/kaggle/input/datasets/phhasian0710/seed-iv/ReadMe.txt
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/4_20151118.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/14_20151208.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/5_20160413.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/3_20151018.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/6_20150511.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/15_20150514.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/10_20151021.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/12_20150804.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/13_20151125.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/11_20150921.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/8_20151110.

In [10]:

"""
This pipeline implements a balanced training approach for EEG emotion recognition
with special focus on handling class imbalance across all 4 emotion classes.
"""

# %% [markdown]
# ## CELL 0: INSTALL DEPENDENCIES

# %%
!pip install mne PyWavelets scikit-learn tqdm matplotlib seaborn pandas scipy numpy torch torchvision torchaudio imbalanced-learn

# %% [markdown]
# ## CELL 1: IMPORTS AND CONFIGURATION

# CELL 1: IMPORTS AND CONFIGURATION
# ============================================================================

import os
import glob
import time
import math
import pickle
import random
from collections import Counter
from itertools import permutations

import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.stats import entropy, kurtosis, zscore
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import warnings
warnings.filterwarnings('ignore')

# ===== SET SEED FUNCTION =====
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"✓ Seed set to {seed}")

# ===== CONFIGURATION WITH BALANCE OPTIMIZATION =====
class Config:
    # Paths
    DATA_PATH = '/kaggle/input/datasets/phhasian0710/seed-iv/eeg_feature_smooth/'
    OUTPUT_PATH = './processed_data/'
    MODEL_PATH = './saved_models/'
    RESULTS_PATH = './results/'
    
    # Dataset parameters
    NUM_SUBJECTS = 15
    NUM_CLASSES = 4
    NUM_CHANNELS = 62
    NUM_SESSIONS = 3
    TRIALS_PER_SESSION = 24
    
    # Emotion mapping
    TRIAL_EMOTION_MAPPING = [1, 2, 3, 0, 2, 0, 0, 1, 0, 1, 2, 1,
                             1, 1, 2, 3, 2, 2, 3, 3, 0, 3, 0, 3]
    EMOTION_NAMES = {0: 'Neutral', 1: 'Sad', 2: 'Fear', 3: 'Happy'}
    
    # Feature extraction
    FEATURE_TYPES = ['DE', 'PSD', 'SE', 'PE']
    NUM_FREQ_BANDS = 5
    TOTAL_FEATURES = len(FEATURE_TYPES) * NUM_FREQ_BANDS
    
    # Temporal parameters
    TARGET_TIME_LENGTH = 35
    
    # ===== UPDATED BALANCE OPTIMIZATION SETTINGS WITH ALL FIXES =====
    # Class weights (adjusted based on your results - Neutral and Fear need more weight)
    CLASS_WEIGHTS = [1.3, 0.9, 1.4, 0.4]  # Neutral↑, Fear↑, Happy↓
    
    # Sampling strategy - FIX #2 (more gradient steps)
    USE_BALANCED_SAMPLING = True
    SAMPLES_PER_CLASS = 16  # Reduced from 32 to increase batches per epoch
    BATCH_SIZE = 64  # 16 * 4 = 64 (reduced from 128)
    
    # Loss function enhancements - FIX #5 & #8
    USE_FOCAL_LOSS = True
    FOCAL_GAMMA = 3.0
    FOCAL_GAMMA_START = 1.0  # Start with gamma=1.0, warm up to 3.0
    FOCAL_ALPHA = [0.35, 0.20, 0.35, 0.10]
    LABEL_SMOOTHING = 0.05  # Reduced from 0.15
    
    # Data augmentation for balance
    USE_AUGMENTATION = True
    AUGMENTATION_TYPES = ['noise', 'mixup', 'cutmix', 'time_warp', 'channel_drop', 'hard_mining']
    NOISE_STD = 0.03
    MIXUP_ALPHA = 0.25
    CUTMIX_PROB = 0.6
    
    # Contrastive learning - FIX #4
    USE_CONTRASTIVE = True
    CONTRASTIVE_TEMPERATURE = 0.25
    CONTRASTIVE_WEIGHT = 0.25
    CONTRASTIVE_WARMUP_EPOCHS = 50  # Warm up contrastive loss over 50 epochs
    
    # Hybrid model settings
    D_MODEL = 128
    N_HEADS = 8
    N_TRANSFORMER_LAYERS = 3
    USE_HYBRID = True
    
    # Model architecture
    USE_MULTI_SCALE = True
    SCALE_KERNELS = [3, 5, 7]
    
    USE_REGION_AWARE = True
    REGION_MAPPING = {
        'frontal': list(range(1, 15)),
        'central': list(range(15, 25)),
        'parietal': list(range(25, 37)),
        'temporal': list(range(37, 49)),
        'occipital': list(range(49, 63))
    }
    
    # Training settings - FIX #3 & #6
    USE_ATTENTION = False
    USE_TEMPORAL = True
    DROPOUT = 0.3
    HIDDEN_DIM = 64
    TOPK_GRAPH = 8
    
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 0.0001
    EPOCHS = 350  # Increased from 200
    PATIENCE = 40  # Increased from 30
    SEED = 42
    
    # Scheduler - FIX #3
    USE_PLATEAU_SCHEDULER = True  # Use ReduceLROnPlateau instead of CosineAnnealing
    
    # Monitoring
    PRINT_FREQUENCY = 10
    SAVE_BEST_ONLY = True

# Create directories
for path in [Config.OUTPUT_PATH, Config.MODEL_PATH, Config.RESULTS_PATH]:
    os.makedirs(path, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(Config.SEED)

print("="*60)
print("BALANCE-OPTIMIZED CONFIGURATION (UPDATED WITH ALL FIXES)")
print("="*60)
print(f"Device: {device}")
print(f"Class weights: {Config.CLASS_WEIGHTS}")
print(f"Focal alpha: {Config.FOCAL_ALPHA}")
print(f"Focal gamma: {Config.FOCAL_GAMMA} (warmup from {Config.FOCAL_GAMMA_START})")
print(f"Label smoothing: {Config.LABEL_SMOOTHING}")
print(f"Batch size: {Config.BATCH_SIZE} ({Config.SAMPLES_PER_CLASS} per class)")
print(f"Contrastive weight: {Config.CONTRASTIVE_WEIGHT} (warmup over {Config.CONTRASTIVE_WARMUP_EPOCHS} epochs)")
print(f"Augmentation types: {Config.AUGMENTATION_TYPES}")
print(f"Epochs: {Config.EPOCHS}, Patience: {Config.PATIENCE}")
print(f"Scheduler: ReduceLROnPlateau")
print("="*60)

# %% [markdown]
# ## CELL 2: DATA PREPROCESSOR

# %%
class SEEDIVDataPreprocessor:
    """Load and preprocess SEED-IV dataset with balance checking"""
    
    def __init__(self, config):
        self.config = config
        self.trial_order = config.TRIAL_EMOTION_MAPPING
        self.emotion_names = config.EMOTION_NAMES
    
    def load_data(self):
        """Load all .mat files and extract DE features"""
        files = sorted(glob.glob(os.path.join(self.config.DATA_PATH, '*.mat')))
        files.extend(sorted(glob.glob(os.path.join(self.config.DATA_PATH, '*/*.mat'))))
        files = sorted(list(set(files)))
        
        all_data, all_labels, all_subjects = [], [], []
        time_lengths = []
        
        print(f"\nLoading {len(files)} files...")
        
        for file_path in files:
            filename = os.path.basename(file_path)
            try:
                subject_id = int(filename.split('_')[0])
                if subject_id > self.config.NUM_SUBJECTS:
                    continue
                
                data = sio.loadmat(file_path)
                de_keys = [k for k in data.keys() if 'de_movingAve' in k]
                de_keys = sorted(de_keys, key=lambda x: int(''.join(filter(str.isdigit, x)) or 0))
                
                for trial_idx, key in enumerate(de_keys[:24]):
                    trial_data = data[key]
                    
                    # Fix shape issues
                    if trial_data.ndim == 3:
                        if trial_data.shape[0] == 62 and trial_data.shape[1] == 5 and trial_data.shape[2] != 5:
                            trial_data = trial_data.transpose(0, 2, 1)
                        elif trial_data.shape[0] == 5 and trial_data.shape[1] == 62:
                            trial_data = trial_data.transpose(1, 2, 0)
                        elif trial_data.shape[2] == 5 and trial_data.shape[1] == 62:
                            trial_data = trial_data.transpose(1, 0, 2)
                    
                    if trial_data.shape[0] != 62 or trial_data.shape[2] != 5:
                        continue
                    
                    time_lengths.append(trial_data.shape[1])
                    all_data.append(trial_data)
                    all_labels.append(self.trial_order[trial_idx])
                    all_subjects.append(subject_id)
                    
            except Exception as e:
                print(f"  Error loading {filename}: {e}")
                continue
        
        all_labels = np.array(all_labels)
        all_subjects = np.array(all_subjects)
        
        # Check class balance
        unique, counts = np.unique(all_labels, return_counts=True)
        balance_ratio = max(counts) / min(counts)
        
        print(f"\n✓ Loaded {len(all_data)} trials")
        print(f"  Time range: {min(time_lengths)}-{max(time_lengths)} points")
        print(f"  Labels: {dict(zip(unique, counts))}")
        print(f"  Balance ratio (max/min): {balance_ratio:.2f}")
        
        return all_data, all_labels, all_subjects
    
    def standardize_time(self, data_list):
        """Make all trials same length"""
        target = self.config.TARGET_TIME_LENGTH
        
        standardized = []
        for trial in data_list:
            current = trial.shape[1]
            
            if current >= target:
                start = (current - target) // 2
                end = start + target
                standardized_trial = trial[:, start:end, :]
            else:
                pad_before = (target - current) // 2
                pad_after = target - current - pad_before
                standardized_trial = np.pad(
                    trial,
                    ((0, 0), (pad_before, pad_after), (0, 0)),
                    mode='constant',
                    constant_values=0
                )
            
            standardized.append(standardized_trial)
        
        result = np.stack(standardized, axis=0)
        print(f"\n⏱️ Time standardized to {target}: {result.shape}")
        return result
    
    def normalize_per_subject(self, data, subjects):
        """Normalize each subject's data separately"""
        result = np.zeros_like(data, dtype=np.float32)
        unique_subjects = np.unique(subjects)
        
        for subject in unique_subjects:
            mask = subjects == subject
            subject_data = data[mask]
            original_shape = subject_data.shape
            flat = subject_data.reshape(-1, original_shape[-1])
            
            scaler = StandardScaler()
            normalized = scaler.fit_transform(flat)
            result[mask] = normalized.reshape(original_shape)
        
        print(f"\n📊 Normalized: mean={result.mean():.4f}, std={result.std():.4f}")
        return result

# %% [markdown]
# ## CELL 3: MULTI-FEATURE EXTRACTOR

# %%
class MultiFeatureExtractor:
    """Extract multiple features from DE data"""
    
    def __init__(self, config):
        self.config = config
        self.feature_types = config.FEATURE_TYPES
    
    def compute_sample_entropy(self, data):
        """Sample Entropy approximation"""
        samples, channels, time, bands = data.shape
        result = np.zeros_like(data, dtype=np.float32)
        window = min(10, time)
        
        for s in range(samples):
            for c in range(channels):
                for b in range(bands):
                    ts = data[s, c, :, b]
                    for t in range(time):
                        start = max(0, t - window)
                        segment = ts[start:t+1]
                        if len(segment) >= 2:
                            local_var = np.var(segment) + 1e-8
                            global_var = np.var(ts) + 1e-8
                            result[s, c, t, b] = -np.log(local_var / global_var + 1e-8)
        return result
    
    def compute_permutation_entropy(self, data, order=3):
        """Permutation Entropy"""
        samples, channels, time, bands = data.shape
        result = np.zeros_like(data, dtype=np.float32)
        perms = list(set(permutations(range(order))))
        n_perms = len(perms)
        
        for s in range(samples):
            for c in range(channels):
                for b in range(bands):
                    ts = data[s, c, :, b]
                    for t in range(time-order+1):
                        pattern = tuple(ts[t:t+order])
                        ranked = tuple(np.argsort(np.argsort(pattern)))
                        if ranked in perms:
                            result[s, c, t:t+order, b] += 0.1
        return result / (np.log(n_perms) + 1e-8)
    
    def extract_features(self, de_data):
        """Combine all features"""
        all_features = []
        
        if 'DE' in self.feature_types:
            all_features.append(de_data)
        
        if 'PSD' in self.feature_types:
            psd = np.exp(np.clip(de_data, -5, 5))
            all_features.append(psd)
        
        if 'SE' in self.feature_types:
            se = self.compute_sample_entropy(de_data)
            se = (se - se.mean()) / (se.std() + 1e-8)
            all_features.append(se)
        
        if 'PE' in self.feature_types:
            pe = self.compute_permutation_entropy(de_data)
            pe = (pe - pe.mean()) / (pe.std() + 1e-8)
            all_features.append(pe)
        
        result = np.concatenate(all_features, axis=-1)
        print(f"\n✓ Multi-features extracted: {result.shape}")
        return result

# %% [markdown]
# ## CELL 4: BALANCED DATASET AND SAMPLER

# %%
class EEGEmotionDataset(Dataset):
    """PyTorch Dataset for EEG emotion recognition"""
    
    def __init__(self, data, labels, name=""):
        self.data = torch.FloatTensor(data)
        self.labels = torch.LongTensor(labels)
        print(f"\n{name} Dataset: {len(self.data)} samples")
        print(f"  Shape: {self.data.shape}")
        print(f"  Class distribution: {np.bincount(labels)}")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

class BalancedBatchSampler:
    """Samples batches with equal representation from all classes"""
    
    def __init__(self, labels, batch_size, num_classes):
        self.labels = np.array(labels)
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.samples_per_class = batch_size // num_classes
        
        # Create indices for each class
        self.class_indices = []
        self.class_counts = []
        
        for c in range(num_classes):
            idx = np.where(self.labels == c)[0]
            np.random.shuffle(idx)
            self.class_indices.append(idx)
            self.class_counts.append(len(idx))
        
        # Number of batches per epoch
        min_class_samples = min(self.class_counts)
        self.num_batches = min_class_samples // self.samples_per_class
        
        print(f"\n📊 Balanced sampler created:")
        print(f"  Samples per class per batch: {self.samples_per_class}")
        print(f"  Batches per epoch: {self.num_batches}")
        print(f"  Total samples per epoch: {self.num_batches * batch_size}")
    
    def __iter__(self):
        # Shuffle indices at start of each epoch
        class_pos = [0] * self.num_classes
        for c in range(self.num_classes):
            np.random.shuffle(self.class_indices[c])
        
        for _ in range(self.num_batches):
            batch = []
            
            # Sample equally from each class
            for c in range(self.num_classes):
                indices = self.class_indices[c]
                start = class_pos[c]
                
                # If not enough samples left, reshuffle
                if start + self.samples_per_class > len(indices):
                    np.random.shuffle(indices)
                    start = 0
                    class_pos[c] = self.samples_per_class
                else:
                    class_pos[c] += self.samples_per_class
                
                batch.extend(indices[start:start + self.samples_per_class])
            
            np.random.shuffle(batch)
            yield batch
    
    def __len__(self):
        return self.num_batches

class WeightedSampler:
    """Alternative: Weighted random sampler for imbalanced data"""
    
    def __init__(self, labels, num_classes, config):
        self.labels = np.array(labels)
        self.num_classes = num_classes
        
        # Calculate class weights
        class_counts = np.bincount(labels, minlength=num_classes)
        class_weights = 1.0 / (class_counts + 1e-8)
        class_weights = class_weights / class_weights.sum()
        
        # Assign weight to each sample
        sample_weights = class_weights[labels]
        
        self.sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(labels) * 2,  # Oversample
            replacement=True
        )
        
        print(f"\n📊 Weighted sampler created:")
        print(f"  Class weights: {class_weights}")
    
    def get_sampler(self):
        return self.sampler

# %% [markdown]
# ## CELL 5: ENHANCED LOSS FUNCTIONS WITH BALANCE

# %%
class FocalLoss(nn.Module):
    """Focal Loss with class weights and label smoothing"""
    
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.1, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing
        self.reduction = reduction
        
    def forward(self, inputs, targets):
        # Label smoothing
        if self.label_smoothing > 0:
            num_classes = inputs.size(-1)
            smooth_targets = torch.zeros_like(inputs)
            smooth_targets.fill_(self.label_smoothing / (num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1 - self.label_smoothing)
            
            log_probs = F.log_softmax(inputs, dim=-1)
            ce_loss = -(smooth_targets * log_probs).sum(dim=-1)
        else:
            ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        
        pt = torch.exp(-ce_loss)
        
        # Apply class weights
        if self.alpha is not None:
            if isinstance(self.alpha, (list, tuple)):
                alpha = torch.tensor(self.alpha).to(inputs.device)
                alpha_t = alpha[targets]
            else:
                alpha_t = self.alpha[targets]
            focal_loss = alpha_t * (1 - pt) ** self.gamma * ce_loss
        else:
            focal_loss = (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

class ContrastiveLoss(nn.Module):
    """NT-Xent loss for contrastive learning with class-aware sampling"""
    
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, features, labels=None):
        # Normalize features
        features = F.normalize(features, dim=1)
        
        # Compute similarity matrix
        similarity = torch.matmul(features, features.T) / self.temperature
        
        # Create labels (diagonal is positive pairs)
        batch_size = features.size(0)
        
        if labels is not None:
            # Create positive pairs from same class
            labels = labels.view(-1, 1)
            mask = torch.eq(labels, labels.T).float()
            mask = mask - torch.eye(batch_size).to(mask.device)  # Remove self
            pos_weight = mask.sum(dim=1)
            
            # Weight positives by class frequency
            pos_similarity = (similarity * mask).sum(dim=1) / (pos_weight + 1e-8)
            neg_similarity = similarity * (1 - mask - torch.eye(batch_size).to(mask.device))
            
            loss = -torch.log(
                torch.exp(pos_similarity) / 
                (torch.exp(pos_similarity) + neg_similarity.exp().sum(dim=1))
            ).mean()
        else:
            # Standard NT-Xent
            labels = torch.arange(batch_size).to(features.device)
            loss = F.cross_entropy(similarity, labels)
        
        return loss

✓ Seed set to 42
BALANCE-OPTIMIZED CONFIGURATION (UPDATED WITH ALL FIXES)
Device: cuda
Class weights: [1.3, 0.9, 1.4, 0.4]
Focal alpha: [0.35, 0.2, 0.35, 0.1]
Focal gamma: 3.0 (warmup from 1.0)
Label smoothing: 0.05
Batch size: 64 (16 per class)
Contrastive weight: 0.25 (warmup over 50 epochs)
Augmentation types: ['noise', 'mixup', 'cutmix', 'time_warp', 'channel_drop', 'hard_mining']
Epochs: 350, Patience: 40
Scheduler: ReduceLROnPlateau


In [11]:
# ## CELL 6: BALANCED DATA AUGMENTATION

# %%
class BalancedAugmentation:
    """Multi-strategy augmentation with class balance consideration"""
    
    def __init__(self, config):
        self.config = config
        self.noise_std = config.NOISE_STD
        self.mixup_alpha = config.MIXUP_ALPHA
        self.cutmix_prob = config.CUTMIX_PROB
        self.training = True
    
    def train(self):
        self.training = True
    
    def eval(self):
        self.training = False
    
    def gaussian_noise(self, x):
        """Add Gaussian noise"""
        noise = torch.randn_like(x) * self.noise_std
        return x + noise
    
    def mixup(self, x, y):
        """Mixup augmentation"""
        batch_size = x.size(0)
        index = torch.randperm(batch_size).to(x.device)
        
        lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)
        lam = max(lam, 1 - lam)
        
        mixed_x = lam * x + (1 - lam) * x[index]
        
        if y.dim() == 1:
            y_onehot = F.one_hot(y, num_classes=self.config.NUM_CLASSES).float()
            y_mixed = lam * y_onehot + (1 - lam) * y_onehot[index]
        else:
            y_mixed = lam * y + (1 - lam) * y[index]
        
        return mixed_x, y_mixed
    
    def cutmix(self, x, y):
        """CutMix augmentation for EEG"""
        batch_size = x.size(0)
        index = torch.randperm(batch_size).to(x.device)
        
        # Random cut point in time dimension
        time_dim = x.size(3)  # (B, C, F, T)
        cut_ratio = np.random.beta(1, 1)
        cut_point = int(time_dim * cut_ratio)
        
        # Create mixed sample
        mixed_x = x.clone()
        mixed_x[:, :, :, cut_point:] = x[index, :, :, cut_point:]
        
        # Mix labels
        lam = cut_point / time_dim
        if y.dim() == 1:
            y_onehot = F.one_hot(y, num_classes=self.config.NUM_CLASSES).float()
            y_mixed = lam * y_onehot + (1 - lam) * y_onehot[index]
        else:
            y_mixed = lam * y + (1 - lam) * y[index]
        
        return mixed_x, y_mixed
    
    def time_warp(self, x, y):
        """Time warping augmentation"""
        batch_size, channels, features, time = x.shape
        
        # Random stretch/compress
        warp_factor = 1.0 + np.random.uniform(-0.2, 0.2)
        new_time = int(time * warp_factor)
        
        # Resample using interpolation
        x_reshaped = x.view(batch_size * channels, features, time)
        x_warped = F.interpolate(x_reshaped, size=new_time, mode='linear', align_corners=False)
        
        if new_time > time:
            # Crop if longer
            start = (new_time - time) // 2
            x_warped = x_warped[:, :, start:start+time]
        elif new_time < time:
            # Pad if shorter
            pad = time - new_time
            x_warped = F.pad(x_warped, (pad//2, pad - pad//2))
        
        return x_warped.view(batch_size, channels, features, time), y
    
    def __call__(self, x, y):
        """Apply random augmentation during training"""
        if not self.training:
            return x, y
        
        # 70% chance to apply augmentation
        if np.random.random() > 0.3:
            aug_type = np.random.choice(self.config.AUGMENTATION_TYPES)
            
            if aug_type == 'noise':
                return self.gaussian_noise(x), y
            elif aug_type == 'mixup':
                return self.mixup(x, y)
            elif aug_type == 'cutmix':
                return self.cutmix(x, y)
            elif aug_type == 'time_warp':
                return self.time_warp(x, y)
        
        return x, y

# %% [markdown]
# ## CELL 7: GRAPH NEURAL NETWORK COMPONENTS

# %%
class GraphConvolution(nn.Module):
    """Basic graph convolution layer"""
    
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        self.bias = nn.Parameter(torch.FloatTensor(out_features))
        nn.init.xavier_uniform_(self.weight)
        nn.init.zeros_(self.bias)
    
    def forward(self, x, adj):
        return torch.matmul(adj, torch.matmul(x, self.weight)) + self.bias

class SelfOrganizedGraphConstruction(nn.Module):
    """Self-organized graph construction"""
    
    def __init__(self, in_features, bn_features, out_features, topk=10):
        super().__init__()
        self.topk = topk
        self.bottleneck = nn.Linear(in_features, bn_features)
        self.graph_conv = GraphConvolution(in_features, out_features)
    
    def forward(self, x):
        xa = torch.tanh(self.bottleneck(x))
        adj = F.softmax(torch.matmul(xa, xa.transpose(1, 2)), dim=2)
        
        # Top-k sparsification
        mask = torch.zeros_like(adj)
        _, idx = adj.topk(self.topk, dim=2)
        mask.scatter_(2, idx, 1.0)
        
        return F.relu(self.graph_conv(x, adj * mask))

# CELL 8: BALANCED DATA AUGMENTATION (UPDATED)
# ============================================================================

class BalancedBatchSampler:
    """Samples batches with equal representation from all classes - FIXED with repeat factor"""
    
    def __init__(self, labels, batch_size, num_classes, repeat_factor=4):
        self.labels = np.array(labels)
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.samples_per_class = batch_size // num_classes
        self.repeat_factor = repeat_factor
        
        # Create indices for each class
        self.class_indices = []
        self.class_counts = []
        
        for c in range(num_classes):
            idx = np.where(self.labels == c)[0]
            # Repeat indices to increase effective samples per epoch
            repeated_idx = np.tile(idx, repeat_factor)
            np.random.shuffle(repeated_idx)
            self.class_indices.append(repeated_idx)
            self.class_counts.append(len(repeated_idx))
        
        # Number of batches per epoch (now multiplied by repeat_factor)
        min_class_samples = min(self.class_counts)
        self.num_batches = min_class_samples // self.samples_per_class
        
        print(f"\n📊 Balanced sampler created:")
        print(f"  Samples per class per batch: {self.samples_per_class}")
        print(f"  Batches per epoch: {self.num_batches} (repeat_factor={repeat_factor})")
        print(f"  Total samples per epoch: {self.num_batches * batch_size}")
    
    def __iter__(self):
        # Shuffle indices at start of each epoch
        class_pos = [0] * self.num_classes
        for c in range(self.num_classes):
            np.random.shuffle(self.class_indices[c])
        
        for _ in range(self.num_batches):
            batch = []
            
            # Sample equally from each class
            for c in range(self.num_classes):
                indices = self.class_indices[c]
                start = class_pos[c]
                
                # If not enough samples left, reshuffle
                if start + self.samples_per_class > len(indices):
                    np.random.shuffle(indices)
                    start = 0
                    class_pos[c] = self.samples_per_class
                else:
                    class_pos[c] += self.samples_per_class
                
                batch.extend(indices[start:start + self.samples_per_class])
            
            np.random.shuffle(batch)
            yield batch
    
    def __len__(self):
        return self.num_batches
        
# CELL 9: LOSS FUNCTIONS AND CONTRASTIVE LEARNING (UPDATED)
# ============================================================================

class AdaptiveFocalLoss(nn.Module):
    """Focal Loss that adapts weights based on training progress - FIXED with gamma warmup"""
    
    def __init__(self, gamma=3.0, gamma_start=1.0, alpha=None, label_smoothing=0.05, reduction='mean'):
        super().__init__()
        self.target_gamma = gamma
        self.current_gamma = gamma_start
        self.initial_alpha = alpha
        self.label_smoothing = label_smoothing
        self.reduction = reduction
        self.alpha = alpha
        self.epoch = 0
        self.class_acc_history = []
        self.warmup_epochs = 50
        
    def update_weights(self, class_accuracies):
        """Dynamically adjust weights based on per-class validation performance"""
        if class_accuracies is not None:
            self.class_acc_history.append(class_accuracies)
            
            # Increase gamma gradually over warmup_epochs
            self.epoch += 1
            if self.epoch <= self.warmup_epochs:
                progress = self.epoch / self.warmup_epochs
                self.current_gamma = self.current_gamma + (self.target_gamma - self.current_gamma) * progress
            
            # Use recent accuracy (last 5 epochs) for stability
            if len(self.class_acc_history) >= 5:
                recent_accs = np.mean(self.class_acc_history[-5:], axis=0)
            else:
                recent_accs = np.array(class_accuracies)
            
            # Calculate weights: lower accuracy = higher weight
            # Add small epsilon to avoid division by zero
            weights = 1.0 / (recent_accs + 5.0)  # +5 to smooth
            
            # Normalize to sum to number of classes
            weights = weights / weights.sum() * len(weights)
            
            # Blend with initial alpha (70% adaptive, 30% initial)
            if self.initial_alpha is not None:
                initial = np.array(self.initial_alpha)
                self.alpha = (0.7 * weights + 0.3 * initial).tolist()
            else:
                self.alpha = weights.tolist()
            
            if self.epoch % 10 == 0:
                print(f"Updated class weights: Neu={self.alpha[0]:.2f}, "
                      f"Sad={self.alpha[1]:.2f}, Fear={self.alpha[2]:.2f}, "
                      f"Happy={self.alpha[3]:.2f} (gamma={self.current_gamma:.2f})")
    
    def forward(self, inputs, targets):
        # Label smoothing
        if self.label_smoothing > 0:
            num_classes = inputs.size(-1)
            smooth_targets = torch.zeros_like(inputs)
            smooth_targets.fill_(self.label_smoothing / (num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1 - self.label_smoothing)
            
            log_probs = F.log_softmax(inputs, dim=-1)
            ce_loss = -(smooth_targets * log_probs).sum(dim=-1)
        else:
            ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        
        pt = torch.exp(-ce_loss)
        
        # Apply class weights with current gamma
        if self.alpha is not None:
            alpha_tensor = torch.tensor(self.alpha).to(inputs.device)
            alpha_t = alpha_tensor[targets]
            focal_loss = alpha_t * (1 - pt) ** self.current_gamma * ce_loss
        else:
            focal_loss = (1 - pt) ** self.current_gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss
        
# CELL 10: FINAL FIXED HYBRID SOGNN-TRANSFORMER MODEL
# ============================================================================

class PositionalEncoding(nn.Module):
    """Learnable positional encoding for temporal dimension"""
    
    def __init__(self, d_model, max_len=35):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, max_len, d_model))
        
    def forward(self, x):
        # x: (batch, channels, time, d_model)
        return x + self.pos_embedding[:, :x.size(2), :].unsqueeze(1)


class TemporalPyramidModule(nn.Module):
    """Multi-scale temporal feature extraction"""
    
    def __init__(self, d_model, scales=[3, 5, 7, 9]):
        super().__init__()
        self.scales = scales
        self.convs = nn.ModuleList()
        
        for k in scales:
            self.convs.append(
                nn.Sequential(
                    nn.Conv1d(d_model, d_model, kernel_size=k, padding=k//2),
                    nn.BatchNorm1d(d_model),
                    nn.ReLU(),
                    nn.Conv1d(d_model, d_model, kernel_size=k, padding=k//2),
                    nn.BatchNorm1d(d_model),
                    nn.ReLU()
                )
            )
        
        self.fusion = nn.Linear(d_model * len(scales), d_model)
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, x):
        # x: (batch*channels, d_model, time)
        multi_scale_features = []
        
        for conv in self.convs:
            feat = conv(x)
            multi_scale_features.append(feat)
        
        # Concatenate along feature dimension
        combined = torch.cat(multi_scale_features, dim=1)  # (N, d_model*num_scales, time)
        
        # Permute for linear layer
        combined = combined.permute(0, 2, 1)  # (N, time, d_model*num_scales)
        fused = self.fusion(combined)  # (N, time, d_model)
        fused = self.norm(fused)
        
        return fused.permute(0, 2, 1)  # (N, d_model, time)


class InterCorticalAttention(nn.Module):
    """Models interactions between brain regions using multi-head attention"""
    
    def __init__(self, d_model, n_heads=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        
        assert self.head_dim * n_heads == d_model, "d_model must be divisible by n_heads"
        
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)
        
        # Region-specific biases (based on brain anatomy)
        self.register_buffer(
            'region_bias',
            torch.ones(62, 62) * 0.1
        )
        
    def forward(self, x, region_mask=None):
        # x: (batch, channels, d_model)
        batch, channels, d_model = x.shape
        
        # Project to Q, K, V
        Q = self.q_proj(x).view(batch, channels, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(batch, channels, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(batch, channels, self.n_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # Add region bias
        if region_mask is not None:
            scores = scores + region_mask.unsqueeze(0).unsqueeze(0)
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        # Apply attention to values
        out = torch.matmul(attn, V).transpose(1, 2).contiguous()
        out = out.view(batch, channels, d_model)
        out = self.out_proj(out)
        
        # Residual connection and layer norm
        out = self.norm(x + out)
        
        return out, attn


class CrossDomainFusion(nn.Module):
    """Adaptive fusion of time and frequency domains"""
    
    def __init__(self, d_model, num_freq_bands=5):
        super().__init__()
        self.num_freq_bands = num_freq_bands
        self.d_model = d_model
        
        # Learnable fusion weights per frequency band
        self.freq_weights = nn.Parameter(torch.ones(num_freq_bands) / num_freq_bands)
        
    def forward(self, x):
        # x: (batch, channels, time, d_model)
        batch, channels, time, d_model = x.shape
        
        # Create multiple views of the data for different "frequency bands"
        band_features = []
        
        # Band 1: Mean across channels
        band1 = x.mean(dim=1)  # (batch, time, d_model)
        band1 = band1.mean(dim=-1)  # (batch, time)
        band_features.append(band1)
        
        # Band 2: Max across channels
        band2 = x.max(dim=1)[0]  # (batch, time, d_model)
        band2 = band2.mean(dim=-1)  # (batch, time)
        band_features.append(band2)
        
        # Band 3: Std across channels
        band3 = x.std(dim=1)  # (batch, time, d_model)
        band3 = band3.mean(dim=-1)  # (batch, time)
        band_features.append(band3)
        
        # Band 4: Min across channels
        band4 = x.min(dim=1)[0]  # (batch, time, d_model)
        band4 = band4.mean(dim=-1)  # (batch, time)
        band_features.append(band4)
        
        # Band 5: L2 norm across channels
        band5 = torch.norm(x, dim=1)  # (batch, time, d_model)
        band5 = band5.mean(dim=-1)  # (batch, time)
        band_features.append(band5)
        
        # Stack bands
        band_stack = torch.stack(band_features[:self.num_freq_bands], dim=-1)  # (batch, time, num_bands)
        
        # Apply learnable frequency weights
        freq_weights = F.softmax(self.freq_weights, dim=0)
        
        # Weighted sum across bands
        freq_fused = torch.einsum('btf,f->bt', band_stack, freq_weights)  # (batch, time)
        
        # Expand to match expected output shape (batch, channels, time)
        freq_fused = freq_fused.unsqueeze(1).expand(-1, channels, -1)  # (batch, channels, time)
        
        return freq_fused  # (batch, channels, time)


class SubjectAdaptationLayer(nn.Module):
    """Subject-specific feature adaptation"""
    
    def __init__(self, d_model, num_subjects=15):
        super().__init__()
        self.num_subjects = num_subjects
        
        # Subject-specific embeddings
        self.subject_embeddings = nn.Embedding(num_subjects + 1, d_model)
        
        # Adaptation networks
        self.scale = nn.Parameter(torch.ones(1))
        self.shift = nn.Parameter(torch.zeros(1))
        
        # Subject-specific gates
        self.subject_gates = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, 1),
                nn.Sigmoid()
            ) for _ in range(num_subjects)
        ])
        
    def forward(self, x, subject_ids=None):
        # x: (batch, channels, time)
        
        if subject_ids is not None:
            # Get subject embeddings
            subj_embed = self.subject_embeddings(subject_ids)  # (batch, d_model)
            
            # Apply subject-specific gating
            batch_gates = []
            for i, subj_id in enumerate(subject_ids):
                if subj_id < self.num_subjects:
                    gate = self.subject_gates[subj_id](subj_embed[i])
                else:
                    gate = torch.ones(1, device=x.device)
                batch_gates.append(gate)
            
            gates = torch.stack(batch_gates)  # (batch, 1)
            
            # Apply adaptation
            x = x * (self.scale * gates.unsqueeze(-1).unsqueeze(-1)) + self.shift
        
        return x


class HybridSOGNNTransformer(nn.Module):
    """
    Integrated Hybrid Model combining SOGNN with Transformer architecture - FINAL FIXED VERSION
    """
    
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.channels = config.NUM_CHANNELS
        self.num_features = config.TOTAL_FEATURES
        self.num_classes = config.NUM_CLASSES
        self.num_subjects = config.NUM_SUBJECTS
        
        # Model dimensions
        self.d_model = 128  # Transformer dimension
        self.n_heads = 8
        self.num_freq_bands = config.NUM_FREQ_BANDS
        self.graph_out_dim = 32  # Graph output dimension
        
        print("\n" + "="*60)
        print("BUILDING HYBRID SOGNN-TRANSFORMER MODEL")
        print("="*60)
        
        # 1. INPUT PROJECTION LAYER
        self.input_projection = nn.Linear(
            self.num_features, self.d_model
        )
        print(f"✓ Input projection: {self.num_features} → {self.d_model}")
        
        # 2. POSITIONAL ENCODING
        self.pos_encoding = PositionalEncoding(self.d_model, config.TARGET_TIME_LENGTH)
        print(f"✓ Positional encoding: time={config.TARGET_TIME_LENGTH}")
        
        # 3. TEMPORAL PYRAMID MODULE
        self.temporal_pyramid = TemporalPyramidModule(
            self.d_model, scales=[3, 5, 7, 9]
        )
        print(f"✓ Temporal pyramid: scales=[3,5,7,9]")
        
        # 4. INTER-CORTICAL ATTENTION (3 layers)
        self.attention_layers = nn.ModuleList([
            InterCorticalAttention(self.d_model, self.n_heads, config.DROPOUT)
            for _ in range(3)
        ])
        print(f"✓ Inter-cortical attention: 3 layers, {self.n_heads} heads")
        
        # Create region mask for brain anatomy
        self.register_buffer('region_mask', self._create_region_mask())
        
        # 5. CROSS-DOMAIN FUSION
        self.cross_domain_fusion = CrossDomainFusion(
            self.d_model, self.num_freq_bands
        )
        print(f"✓ Cross-domain fusion: {self.num_freq_bands} frequency bands")
        
        # 6. SUBJECT ADAPTATION LAYER
        self.subject_adaptation = SubjectAdaptationLayer(
            self.d_model, self.num_subjects
        )
        print(f"✓ Subject adaptation: {self.num_subjects} subjects")
        
        # 7. CNN LAYERS for SOGNN pathway
        self.conv1 = nn.Conv2d(1, 16, kernel_size=(self.num_features, 5))
        self.bn1 = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d((1, 2))
        
        # 8. GRAPH MODULES
        self.graph1 = SelfOrganizedGraphConstruction(
            self._get_conv_output_size(), 32, self.graph_out_dim, config.TOPK_GRAPH
        )
        
        # 9. REGION-AWARE PROCESSORS
        if config.USE_REGION_AWARE:
            self.region_processor1 = RegionAwareProcessor(config, self._get_conv_output_size())
        
        # 10. TRANSFORMER FEATURE PROJECTION - Project transformer features to match graph dimension
        self.transformer_proj = nn.Linear(1, self.graph_out_dim)  # Project from 1 to 32
        
        # 11. FUSION LAYER - Now takes 2*graph_out_dim = 64 features
        fusion_input_dim = self.graph_out_dim * 2  # 64
        self.fusion = nn.Sequential(
            nn.Linear(fusion_input_dim, self.d_model),
            nn.LayerNorm(self.d_model),
            nn.ReLU(),
            nn.Dropout(config.DROPOUT)
        )
        
        # 12. CLASSIFIER
        self.classifier = nn.Sequential(
            nn.Linear(self.d_model, self.d_model // 2),
            nn.LayerNorm(self.d_model // 2),
            nn.ReLU(),
            nn.Dropout(config.DROPOUT),
            nn.Linear(self.d_model // 2, self.num_classes)
        )
        
        # Initialize classifier bias based on class weights
        if hasattr(config, 'CLASS_WEIGHTS'):
            with torch.no_grad():
                weights = torch.tensor(config.CLASS_WEIGHTS)
                self.classifier[-1].bias.data = torch.log(weights / weights.sum())
        
        total_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"\n✓ Total trainable parameters: {total_params:,}")
        print("="*60 + "\n")
    
    def _get_conv_output_size(self):
        """Calculate output size after conv layers"""
        # After conv1 and pool1 with input (batch, 1, features, time)
        # conv1: kernel=(features,5) -> output channels=16, time reduced by 4
        # pool1: kernel=(1,2) -> time halved
        # Original time=35 -> after conv1: time=31 -> after pool1: time=15
        return 16 * 15  # 240
    
    def _create_region_mask(self):
        """Create attention bias based on brain regions"""
        mask = torch.zeros(62, 62)
        
        # Define region groups (based on standard 10-20 system)
        regions = {
            'frontal': list(range(0, 14)),
            'central': list(range(14, 24)),
            'parietal': list(range(24, 36)),
            'temporal': list(range(36, 48)),
            'occipital': list(range(48, 62))
        }
        
        # Intra-region connections get positive bias
        for region, channels in regions.items():
            for i in channels:
                for j in channels:
                    if i != j:
                        mask[i, j] += 0.2
        
        # Inter-region connections (closer regions get higher bias)
        region_order = ['frontal', 'central', 'parietal', 'occipital', 'temporal']
        for i, r1 in enumerate(region_order):
            for j, r2 in enumerate(region_order):
                if i != j:
                    # Bias based on anatomical proximity
                    distance = abs(i - j)
                    bias = 0.1 / (distance + 1)
                    for c1 in regions[r1]:
                        for c2 in regions[r2]:
                            mask[c1, c2] += bias
        
        return mask
    
    def forward(self, x, subject_ids=None, return_features=False):
        """
        x: (batch, channels=62, time=35, features=20)
        subject_ids: (batch,) optional subject identifiers
        """
        batch_size = x.size(0)
        
        # ===== TRANSFORMER PATHWAY =====
        
        # 1. Input projection
        x_proj = self.input_projection(x)  # (batch, channels, time, d_model)
        
        # 2. Add positional encoding
        x_pos = self.pos_encoding(x_proj)
        
        # 3. Temporal pyramid
        x_temp = x_pos.permute(0, 1, 3, 2)  # (batch, channels, d_model, time)
        x_temp_flat = x_temp.reshape(-1, self.d_model, x_temp.size(-1))
        x_temp_pyramid = self.temporal_pyramid(x_temp_flat)
        x_temp_pyramid = x_temp_pyramid.view(batch_size, self.channels, self.d_model, -1)
        x_temp_pyramid = x_temp_pyramid.permute(0, 1, 3, 2)  # (batch, channels, time, d_model)
        
        # 4. Inter-cortical attention
        x_time_avg = x_temp_pyramid.mean(dim=2)  # (batch, channels, d_model)
        
        attention_maps = []
        for attn_layer in self.attention_layers:
            x_time_avg, attn = attn_layer(x_time_avg, self.region_mask)
            attention_maps.append(attn)
        
        x_attended = x_time_avg.unsqueeze(2).expand(-1, -1, x_temp_pyramid.size(2), -1)
        
        # 5. Cross-domain fusion
        x_fused = self.cross_domain_fusion(x_attended)  # (batch, channels, time)
        
        # 6. Subject adaptation
        if subject_ids is not None:
            x_fused = self.subject_adaptation(x_fused, subject_ids)
        
        # Aggregate over time and channels for transformer features
        x_transformer = x_fused.mean(dim=(1, 2))  # (batch) - 1D tensor [128]
        
        # ===== SOGNN PATHWAY =====
        
        # Prepare input for CNN
        x_cnn = x.permute(0, 1, 3, 2)  # (batch, channels, features, time)
        x_cnn = x_cnn.reshape(-1, 1, self.num_features, x.size(2))
        
        # CNN feature extraction
        x_cnn = F.relu(self.bn1(self.conv1(x_cnn)))
        x_cnn = self.pool1(x_cnn)
        x_cnn = x_cnn.view(batch_size, self.channels, -1)  # (batch, channels, 240)
        
        # Region-aware processing
        if self.config.USE_REGION_AWARE and hasattr(self, 'region_processor1'):
            x_cnn = x_cnn + self.region_processor1(x_cnn).unsqueeze(1)
        
        # Graph processing
        x_graph = self.graph1(x_cnn)  # (batch, channels, graph_out_dim=32)
        x_graph = x_graph.mean(dim=1)  # (batch, 32) - 2D tensor [128, 32]
        
        # ===== FUSION - FIXED =====
        # Project transformer features to match graph dimension
        x_transformer = x_transformer.unsqueeze(-1)  # (batch, 1)
        x_transformer_proj = self.transformer_proj(x_transformer)  # (batch, 32)
        
        # Now both are (batch, 32)
        x_combined = torch.cat([x_transformer_proj, x_graph], dim=1)  # (batch, 64)
        
        # Pass through fusion layer
        x_fused = self.fusion(x_combined)  # (batch, d_model=128)
        
        # ===== CLASSIFICATION =====
        output = self.classifier(x_fused)
        
        if return_features:
            return output, x_fused
        
        return output


class SelfOrganizedGraphConstruction(nn.Module):
    """Simplified graph construction for hybrid model"""
    
    def __init__(self, in_features, bn_features, out_features, topk=8):
        super().__init__()
        self.topk = topk
        self.bottleneck = nn.Linear(in_features, bn_features)
        self.graph_conv = GraphConvolution(in_features, out_features)
    
    def forward(self, x):
        xa = torch.tanh(self.bottleneck(x))
        adj = F.softmax(torch.matmul(xa, xa.transpose(1, 2)), dim=2)
        
        mask = torch.zeros_like(adj)
        _, idx = adj.topk(self.topk, dim=2)
        mask.scatter_(2, idx, 1.0)
        
        return F.relu(self.graph_conv(x, adj * mask))


class GraphConvolution(nn.Module):
    """Basic graph convolution"""
    
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        self.bias = nn.Parameter(torch.FloatTensor(out_features))
        nn.init.xavier_uniform_(self.weight)
        nn.init.zeros_(self.bias)
    
    def forward(self, x, adj):
        return torch.matmul(adj, torch.matmul(x, self.weight)) + self.bias


class RegionAwareProcessor(nn.Module):
    """Region-aware processing (simplified for hybrid)"""
    
    def __init__(self, config, input_dim):
        super().__init__()
        self.config = config
        self.regions = config.REGION_MAPPING
        
        self.region_projections = nn.ModuleDict()
        for region_name, channels in self.regions.items():
            channels = [c-1 for c in channels if c-1 < config.NUM_CHANNELS]
            if channels:
                self.region_projections[region_name] = nn.Linear(
                    len(channels) * input_dim, input_dim
                )
        
        self.fusion = nn.Linear(len(self.regions) * input_dim, input_dim)
        self.norm = nn.LayerNorm(input_dim)
    
    def forward(self, x):
        batch_size = x.size(0)
        region_outputs = []
        
        for region_name, channels in self.regions.items():
            channels = [c-1 for c in channels if c-1 < x.size(1)]
            if not channels:
                continue
            
            region_data = x[:, channels, :]
            region_flat = region_data.reshape(batch_size, -1)
            if region_name in self.region_projections:
                region_proj = self.region_projections[region_name](region_flat)
                region_outputs.append(region_proj)
        
        if region_outputs:
            combined = torch.cat(region_outputs, dim=1)
            fused = self.fusion(combined)
            return self.norm(fused + x.mean(dim=1))
        else:
            return x.mean(dim=1)


In [14]:
# CELL 11: BALANCED TRAINER (ALL FIXES APPLIED)
# ============================================================================

class BalancedTrainer:
    """Trainer with class balance optimization and adaptive weighting - ALL FIXES APPLIED"""
    
    def __init__(self, model, train_dataset, val_loader, config, device, fold=None):
        self.model = model
        self.train_dataset = train_dataset
        self.val_loader = val_loader
        self.config = config
        self.device = device
        self.fold = fold
        
        # Get training labels
        train_labels = train_dataset.labels.numpy()
        
        # Create balanced batch sampler with repeat factor (FIX #2)
        if config.USE_BALANCED_SAMPLING:
            self.batch_sampler = BalancedBatchSampler(
                train_labels,
                config.BATCH_SIZE,
                config.NUM_CLASSES,
                repeat_factor=4
            )
            self.train_loader = DataLoader(
                train_dataset,
                batch_sampler=self.batch_sampler,
                num_workers=0
            )
        else:
            sampler = WeightedSampler(train_labels, config.NUM_CLASSES, config)
            self.train_loader = DataLoader(
                train_dataset,
                batch_size=config.BATCH_SIZE,
                sampler=sampler.get_sampler(),
                num_workers=0
            )
        
        # Loss function with adaptive weights and gamma warmup (FIX #5)
        if config.USE_FOCAL_LOSS:
            self.criterion = AdaptiveFocalLoss(
                gamma=config.FOCAL_GAMMA,
                gamma_start=getattr(config, 'FOCAL_GAMMA_START', 1.0),
                alpha=config.FOCAL_ALPHA,
                label_smoothing=config.LABEL_SMOOTHING
            )
            print(f"\n Using Adaptive Focal Loss: gamma={config.FOCAL_GAMMA} (warmup from 1.0)")
        else:
            weights = torch.FloatTensor(config.CLASS_WEIGHTS).to(device)
            self.criterion = nn.CrossEntropyLoss(
                weight=weights,
                label_smoothing=config.LABEL_SMOOTHING
            )
            print(f"\n Using Weighted CE: weights={config.CLASS_WEIGHTS}")
        
        # Optimizer
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config.LEARNING_RATE,
            weight_decay=config.WEIGHT_DECAY
        )
        
        # Scheduler - FIX #3: single definition, no duplicates
        self.use_plateau = getattr(config, 'USE_PLATEAU_SCHEDULER', True)
        if self.use_plateau:
            self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                self.optimizer,
                mode='max',
                factor=0.5,
                patience=10,
                min_lr=1e-6
            )
            print(f" Using ReduceLROnPlateau scheduler")
        else:
            self.scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
                self.optimizer, T_0=50, T_mult=1, eta_min=1e-6
            )
            print(f" Using CosineAnnealingWarmRestarts (T_0=50)")
        
        # Contrastive learning with warmup (FIX #4)
        if config.USE_CONTRASTIVE:
            self.contrastive = ContrastiveLoss(
                temperature=config.CONTRASTIVE_TEMPERATURE
            ).to(device)
            self.contrastive_weight = config.CONTRASTIVE_WEIGHT
            self.contrastive_warmup_epochs = getattr(config, 'CONTRASTIVE_WARMUP_EPOCHS', 50)
            print(f" Using Contrastive Learning: weight={self.contrastive_weight} (warmup over {self.contrastive_warmup_epochs} epochs)")
        
        # Augmentation
        self.augmentation = BalancedAugmentation(config)
        
        # Tracking - FIX #1: track best_combined_score explicitly
        self.best_combined_score = -float('inf')
        self.best_val_acc = 0
        self.best_val_f1 = 0
        self.patience = 0
        self.save_path = os.path.join(config.MODEL_PATH, f'balanced_model_fold{fold}.pth')
        
        # Per-class accuracy history
        self.train_class_acc_history = []
        self.val_class_acc_history = []
        
        # Balance metrics
        self.balance_scores = []
        
        print(f"\n✓ Balanced Trainer ready - Fold {fold}")
        print(f"  Train: {len(train_dataset)} samples")
        print(f"  Val: {len(val_loader.dataset)} samples")
    
    def calculate_balance_score(self, per_class_acc):
        """Calculate how balanced the classes are (0-100)"""
        accs = np.array(per_class_acc)
        mean_acc = np.mean(accs)
        std_acc = np.std(accs)
        balance_score = 100 * (1 - std_acc / (mean_acc + 1e-8))
        return max(0, min(100, balance_score))
    
    def train_epoch(self, epoch):
        self.model.train()
        self.augmentation.train()
        total_loss = 0
        class_correct = [0] * self.config.NUM_CLASSES
        class_total = [0] * self.config.NUM_CLASSES
        
        # FIX #4: contrastive warmup
        if hasattr(self, 'contrastive'):
            warmup_factor = min(1.0, epoch / self.contrastive_warmup_epochs)
            effective_contrastive_weight = self.contrastive_weight * warmup_factor
        else:
            effective_contrastive_weight = 0.0
        
        for batch_x, batch_y in self.train_loader:
            batch_x = batch_x.to(self.device)
            batch_y = batch_y.to(self.device)
            
            # Apply augmentation
            if self.config.USE_AUGMENTATION:
                batch_x, batch_y = self.augmentation(batch_x, batch_y)
            
            self.optimizer.zero_grad()
            
            # Forward pass
            output, features = self.model(batch_x, return_features=True)
            
            # Compute losses
            if batch_y.dim() > 1:  # Mixup/CutMix soft labels
                loss = 0
                for c in range(self.config.NUM_CLASSES):
                    class_weight = batch_y[:, c]
                    if class_weight.sum() > 0:
                        class_loss = self.criterion(
                            output,
                            torch.full_like(batch_y[:, c].long(), c)
                        )
                        loss += class_loss * class_weight.mean()
            else:
                cls_loss = self.criterion(output, batch_y)
                if self.config.USE_CONTRASTIVE and effective_contrastive_weight > 0:
                    contrast_loss = self.contrastive(features, batch_y)
                    loss = cls_loss + effective_contrastive_weight * contrast_loss
                else:
                    loss = cls_loss
            
            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=5.0)
            self.optimizer.step()
            
            total_loss += loss.item()
            
            # Track per-class accuracy
            with torch.no_grad():
                preds = output.argmax(1)
                if batch_y.dim() == 1:
                    for c in range(self.config.NUM_CLASSES):
                        mask = (batch_y == c)
                        class_total[c] += mask.sum().item()
                        class_correct[c] += (preds[mask] == c).sum().item()
        
        per_class_acc = []
        for c in range(self.config.NUM_CLASSES):
            if class_total[c] > 0:
                per_class_acc.append(100 * class_correct[c] / class_total[c])
            else:
                per_class_acc.append(0)
        
        self.train_class_acc_history.append(per_class_acc)
        return total_loss / len(self.train_loader), per_class_acc
    
    def validate(self):
        self.model.eval()
        self.augmentation.eval()
        total_loss = 0
        correct = 0
        total = 0
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for batch_x, batch_y in self.val_loader:
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)
                
                output = self.model(batch_x)
                loss = self.criterion(output, batch_y)
                
                total_loss += loss.item()
                pred = output.argmax(1)
                correct += (pred == batch_y).sum().item()
                total += batch_y.size(0)
                
                all_preds.extend(pred.cpu().numpy())
                all_targets.extend(batch_y.cpu().numpy())
        
        val_acc = 100 * correct / total
        val_f1 = f1_score(all_targets, all_preds, average='macro')
        
        cm = confusion_matrix(all_targets, all_preds)
        per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100
        per_class_acc = np.nan_to_num(per_class_acc, nan=0.0)
        
        self.val_class_acc_history.append(per_class_acc)
        return total_loss / len(self.val_loader), val_acc, val_f1, per_class_acc, all_preds, all_targets
    
    def train(self):
        print(f"\n{'='*60}")
        print(f"TRAINING FOLD {self.fold} (BALANCED - ALL FIXES APPLIED)")
        print(f"{'='*60}")
        
        for epoch in range(self.config.EPOCHS):
            train_loss, per_class_train = self.train_epoch(epoch)
            val_loss, val_acc, val_f1, per_class_val, val_preds, val_targets = self.validate()
            
            # FIX #3: scheduler step - clean, no nesting
            if self.use_plateau:
                old_lr = self.optimizer.param_groups[0]['lr']
                self.scheduler.step(val_f1)
                new_lr = self.optimizer.param_groups[0]['lr']
                if new_lr < old_lr:
                    print(f"  ⬇ LR reduced: {old_lr:.2e} → {new_lr:.2e}")
            else:
                self.scheduler.step()
            
            current_lr = self.optimizer.param_groups[0]['lr']
            
            # Update adaptive weights
            if hasattr(self.criterion, 'update_weights'):
                self.criterion.update_weights(per_class_val)
            
            # Update augmentation hard classes
            if hasattr(self.augmentation, 'update_hard_classes'):
                self.augmentation.update_hard_classes(per_class_val)
            
            # Balance score
            balance_score = self.calculate_balance_score(per_class_val)
            self.balance_scores.append(balance_score)
            
            # Combined score
            combined_score = val_f1 * 100 + balance_score * 0.5
            
            # Print progress
            if epoch % self.config.PRINT_FREQUENCY == 0:
                print(f"\nEpoch {epoch:3d}:")
                print(f"  Train: Neu={per_class_train[0]:.1f}%, Sad={per_class_train[1]:.1f}%, "
                      f"Fear={per_class_train[2]:.1f}%, Happy={per_class_train[3]:.1f}%")
                print(f"  Val:   Neu={per_class_val[0]:.1f}%, Sad={per_class_val[1]:.1f}%, "
                      f"Fear={per_class_val[2]:.1f}%, Happy={per_class_val[3]:.1f}%")
                print(f"  Val Acc={val_acc:.2f}%, Val F1={val_f1:.4f}, Balance={balance_score:.1f}/100, "
                      f"Combined={combined_score:.1f}, LR={current_lr:.2e}")
            
            # FIX #1: compare against stored best_combined_score
            if combined_score > self.best_combined_score:
                self.best_combined_score = combined_score
                self.best_val_f1 = val_f1
                self.best_val_acc = val_acc
                self.patience = 0
                
                if self.config.SAVE_BEST_ONLY:
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': self.model.state_dict(),
                        'optimizer_state_dict': self.optimizer.state_dict(),
                        'val_acc': float(val_acc),
                        'val_f1': float(val_f1),
                        'per_class_val': per_class_val.tolist(),
                        'balance_score': float(balance_score),
                        'combined_score': float(combined_score),
                    }, self.save_path)
                    
                    if epoch % self.config.PRINT_FREQUENCY == 0:
                        print(f"  → New best! (F1={val_f1:.4f}, Balance={balance_score:.1f}, Combined={combined_score:.1f})")
            else:
                self.patience += 1
                if self.patience >= self.config.PATIENCE:
                    print(f"\nEarly stopping at epoch {epoch}")
                    break
        
        print(f"\n✓ Best Validation F1: {self.best_val_f1:.4f} (Acc: {self.best_val_acc:.2f}%)")
        print(f"  Average Balance Score: {np.mean(self.balance_scores):.1f}/100")
        
        return self.best_val_f1

# %% [markdown]
# ## CELL 12: MAIN EXECUTION - LOSO CROSS-VALIDATION

# %%
def run_balanced_loso(data, labels, subjects, config, device):
    """Run Leave-One-Subject-Out cross-validation with balance optimization"""
    
    unique_subjects = np.unique(subjects)
    results = []
    per_class_results = []
    confusion_matrices = []
    
    print("\n" + "="*60)
    print("BALANCED LOSO CROSS-VALIDATION")
    print("="*60)
    
    for fold, test_subj in enumerate(unique_subjects):
        print(f"\n{'='*60}")
        print(f"FOLD {fold+1}/{len(unique_subjects)}: Test Subject {test_subj}")
        print(f"{'='*60}")
        
        # Split data
        test_mask = (subjects == test_subj)
        train_val_mask = ~test_mask
        
        X_test, y_test = data[test_mask], labels[test_mask]
        X_train_val, y_train_val = data[train_val_mask], labels[train_val_mask]
        
        # Stratified validation split
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=config.SEED+fold)
        train_idx, val_idx = next(sss.split(X_train_val, y_train_val))
        X_train, y_train = X_train_val[train_idx], y_train_val[train_idx]
        X_val, y_val = X_train_val[val_idx], y_train_val[val_idx]
        
        # Print class distribution
        train_dist = np.bincount(y_train, minlength=config.NUM_CLASSES)
        val_dist = np.bincount(y_val, minlength=config.NUM_CLASSES)
        test_dist = np.bincount(y_test, minlength=config.NUM_CLASSES)
        
        print(f"\nClass distribution (Neu/Sad/Fear/Happy):")
        print(f"  Train: {train_dist}")
        print(f"  Val:   {val_dist}")
        print(f"  Test:  {test_dist}")
        
        # Calculate class balance ratio
        balance_ratio = max(train_dist) / min(train_dist)
        print(f"  Train balance ratio: {balance_ratio:.2f}")
        
        # Create datasets
        train_dataset = EEGEmotionDataset(X_train, y_train, "Train")
        val_dataset = EEGEmotionDataset(X_val, y_val, "Val")
        test_dataset = EEGEmotionDataset(X_test, y_test, "Test")
        
        # Create data loaders
        val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE)
        test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE)
        
        # Train model
        set_seed(config.SEED + fold)
        model = HybridSOGNNTransformer(config).to(device)

        #model = BalancedSOGNN(config).to(device)
        trainer = BalancedTrainer(model, train_dataset, val_loader, config, device, fold+1)
        best_val_f1 = trainer.train()
        
        # Load best model and evaluate on test set
        checkpoint = torch.load(trainer.save_path, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        model.eval()
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                output = model(batch_x)
                pred = output.argmax(dim=1)
                all_preds.extend(pred.cpu().numpy())
                all_targets.extend(batch_y.cpu().numpy())
        
        all_preds = np.array(all_preds)
        all_targets = np.array(all_targets)
        
        # Calculate metrics
        acc = accuracy_score(all_targets, all_preds) * 100
        f1 = f1_score(all_targets, all_preds, average='macro')
        cm = confusion_matrix(all_targets, all_preds)
        per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100
        
        print(f"\nTest Results - Subject {test_subj}:")
        print(f"  Accuracy: {acc:.2f}%")
        print(f"  F1 Score: {f1:.4f}")
        print(f"  Per-class:")
        print(f"    Neutral: {per_class_acc[0]:.1f}%")
        print(f"    Sad:     {per_class_acc[1]:.1f}%")
        print(f"    Fear:    {per_class_acc[2]:.1f}%")
        print(f"    Happy:   {per_class_acc[3]:.1f}%")
        print(f"  Confusion Matrix:\n{cm}")
        
        results.append({
            'subject': int(test_subj),
            'accuracy': acc,
            'f1': f1,
            'balance_ratio': balance_ratio
        })
        per_class_results.append({
            'subject': int(test_subj),
            'neutral': per_class_acc[0],
            'sad': per_class_acc[1],
            'fear': per_class_acc[2],
            'happy': per_class_acc[3]
        })
        confusion_matrices.append(cm)
        
        # Cleanup
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    # Final summary
    if len(results) > 0:
        print("\n" + "="*60)
        print("FINAL BALANCED LOSO RESULTS")
        print("="*60)
        
        accs = [r['accuracy'] for r in results]
        f1s = [r['f1'] for r in results]
        
        print("\nPer-subject results:")
        for r in results:
            print(f"  Subject {r['subject']:2d}: Acc={r['accuracy']:.2f}%, F1={r['f1']:.4f}")
        
        print(f"\n Overall Performance:")
        print(f"  Mean Accuracy: {np.mean(accs):.2f}% ± {np.std(accs):.2f}%")
        print(f"  Mean F1 Score: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
        
        neutral_accs = [p['neutral'] for p in per_class_results]
        sad_accs = [p['sad'] for p in per_class_results]
        fear_accs = [p['fear'] for p in per_class_results]
        happy_accs = [p['happy'] for p in per_class_results]
        
        print(f"\n Per-class accuracy (mean ± std):")
        print(f"  Neutral: {np.mean(neutral_accs):.2f}% ± {np.std(neutral_accs):.2f}%")
        print(f"  Sad:     {np.mean(sad_accs):.2f}% ± {np.std(sad_accs):.2f}%")
        print(f"  Fear:    {np.mean(fear_accs):.2f}% ± {np.std(fear_accs):.2f}%")
        print(f"  Happy:   {np.mean(happy_accs):.2f}% ± {np.std(happy_accs):.2f}%")
        
        # Calculate balance quality metric
        per_class_means = [np.mean(neutral_accs), np.mean(sad_accs), 
                          np.mean(fear_accs), np.mean(happy_accs)]
        balance_quality = 100 - np.std(per_class_means) * 2  # Higher is better
        print(f"\n⚖️  Balance Quality Score: {balance_quality:.1f}/100")
        
        best_idx = np.argmax(accs)
        worst_idx = np.argmin(accs)
        print(f"\n  Best Subject:  {results[best_idx]['subject']} ({accs[best_idx]:.2f}%)")
        print(f"  Worst Subject: {results[worst_idx]['subject']} ({accs[worst_idx]:.2f}%)")
        
        # Average confusion matrix
        avg_cm = np.mean(confusion_matrices, axis=0)
        print(f"\n Average Confusion Matrix:")
        print(avg_cm.round(1))
        
        print("="*60)
    
    return results, per_class_results

# %% [markdown]
# ## CELL 13: RUN THE PIPELINE

# %%
if __name__ == "__main__":
    print("\n" + "="*60)
    print("BALANCED SOGNN FOR SEED-IV EMOTION RECOGNITION")
    print("WITH CLASS BALANCE OPTIMIZATION")
    print("="*60)
    
    set_seed(Config.SEED)
    print(f"\nUsing device: {device}")
    
    # Load and preprocess data
    print("\n" + "-"*40)
    print("STEP 1: DATA LOADING")
    print("-"*40)
    preprocessor = SEEDIVDataPreprocessor(Config)
    data_list, labels, subjects = preprocessor.load_data()
    data = preprocessor.standardize_time(data_list)
    data = preprocessor.normalize_per_subject(data, subjects)
    
    print("\n" + "-"*40)
    print("STEP 2: FEATURE EXTRACTION")
    print("-"*40)
    extractor = MultiFeatureExtractor(Config)
    data = extractor.extract_features(data)
    
    print(f"\nFinal data shape: {data.shape}")
    print(f"Final labels distribution: {np.bincount(labels)}")
    
    print("\n" + "-"*40)
    print("STEP 3: BALANCED LOSO CROSS-VALIDATION")
    print("-"*40)
    results, per_class_results = run_balanced_loso(data, labels, subjects, Config, device)
    
    # Save results
    print("\n" + "-"*40)
    print("STEP 4: SAVING RESULTS")
    print("-"*40)
    
    results_file = os.path.join(Config.RESULTS_PATH, 'balanced_results.pkl')
    with open(results_file, 'wb') as f:
        pickle.dump({
            'results': results,
            'per_class': per_class_results,
            'config': Config.__dict__
        }, f)
    
    print(f"✓ Results saved to {results_file}")
    
    # Print final summary
    print("\n" + "="*60)
    print("EXPERIMENT COMPLETE")
    print("="*60)


BALANCED SOGNN FOR SEED-IV EMOTION RECOGNITION
WITH CLASS BALANCE OPTIMIZATION
✓ Seed set to 42

Using device: cuda

----------------------------------------
STEP 1: DATA LOADING
----------------------------------------

Loading 45 files...

✓ Loaded 1080 trials
  Time range: 10-64 points
  Labels: {np.int64(0): np.int64(270), np.int64(1): np.int64(270), np.int64(2): np.int64(270), np.int64(3): np.int64(270)}
  Balance ratio (max/min): 1.00

⏱️ Time standardized to 35: (1080, 62, 35, 5)

📊 Normalized: mean=-0.0000, std=1.0000

----------------------------------------
STEP 2: FEATURE EXTRACTION
----------------------------------------

✓ Multi-features extracted: (1080, 62, 35, 20)

Final data shape: (1080, 62, 35, 20)
Final labels distribution: [270 270 270 270]

----------------------------------------
STEP 3: BALANCED LOSO CROSS-VALIDATION
----------------------------------------

BALANCED LOSO CROSS-VALIDATION

FOLD 1/15: Test Subject 1

Class distribution (Neu/Sad/Fear/Happy):
  T

KeyboardInterrupt: 